In [1]:
# If needed (in terminal, not notebook):
# pip install pandas scikit-learn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
import numpy as np


In [2]:
df = pd.read_csv("Cleaned_datasheet_with_awareness_score_and_category2.csv")

df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6168 entries, 0 to 6167
Data columns (total 33 columns):
 #   Column                                                                                        Non-Null Count  Dtype  
---  ------                                                                                        --------------  -----  
 0   if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?              6168 non-null   object 
 1   do_you_know_what_is_a_brain_stroke?                                                           6168 non-null   object 
 2   how_soon_would_you_consult_a_specialist_after_experiencing_the_first_symptom?                 6168 non-null   object 
 3   do_you_think_sudden_confusion_,trouble_speaking_or_understanding_speech_is_a_stroke_symptom?  6168 non-null   object 
 4   do_you_think_sudden_numbness_or_weakness_of_face,_arm_or_leg_is_a_symptom_of_stroke?          6168 non-null   object 
 5   do_you_think_sudden_noseble

In [6]:
target = "awareness_category"

numeric_features = [
    "score_specialist", "score_know_stroke", "score_action_first_symptom",
    "score_confusion", "score_numbness", "score_nosebleed", "score_vision",
    "score_first_contact", "score_urgency", "score_location",
    "score_symptoms", "score_risk", "score_advice",
    "total_raw_score", "awareness_score"
]

categorical_features = [
    "if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?",
    "do_you_know_what_is_a_brain_stroke?",
    "how_soon_would_you_consult_a_specialist_after_experiencing_the_first_symptom?",
    "do_you_think_sudden_confusion_,trouble_speaking_or_understanding_speech_is_a_stroke_symptom?",
    "do_you_think_sudden_numbness_or_weakness_of_face,_arm_or_leg_is_a_symptom_of_stroke?",
    "do_you_think_sudden_nosebleed_is_a_stroke_of_symptom?",
    "do_you_think_trouble_seeing_in_one_or_both_the_eyes_is_a_stroke_symptom?",
    "age", "gender", "educational_level", "salary",
    "first_contact_after_experiencing_symptom",
    "how_soon_treatment_should_be_taken_after_noticing_symptoms",
    "where_to_go_after_experiencing_symptoms_of_brain_stroke"
]

X = df[categorical_features]
y = df[target]


In [8]:
numeric_transformer = "passthrough"

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        # ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

clf = DecisionTreeClassifier(
    random_state=0,
    max_depth=4  # keep small so it’s easy to interpret
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", clf)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

model.fit(X_train, y_train)

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy :", model.score(X_test, y_test))


Train accuracy: 0.9312930685042562
Test accuracy : 0.9286871961102107


In [9]:
# Get underlying tree model
tree = model.named_steps["clf"]

# Get transformed feature names
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(cat_feature_names)

importances = tree.feature_importances_

# Combine and sort
feat_imp = sorted(
    zip(all_feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

for name, imp in feat_imp[:30]:  # top 30
    if imp > 0:
        print(f"{name}: {imp:.3f}")


age_60+: 0.673
gender_Male: 0.141
if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?_Yes: 0.044
first_contact_after_experiencing_symptom_emergency_services: 0.043
score_first_contact: 0.037
score_know_stroke: 0.026
educational_level_undergraduate: 0.017
score_location: 0.013
do_you_know_what_is_a_brain_stroke?_Yes: 0.004
if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?_Physician: 0.000
salary_below 2,00,000: 0.000


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,      # let it grow, forest is more stable
    random_state=0,
    n_jobs=-1
)

rf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", rf_clf)
])

rf_model.fit(X_train, y_train)

print("RF Train accuracy:", rf_model.score(X_train, y_train))
print("RF Test accuracy :", rf_model.score(X_test, y_test))

# Feature importance
ohe = rf_model.named_steps["preprocess"].named_transformers_["cat"]
feature_names = ohe.get_feature_names_out(categorical_features)
importances = rf_model.named_steps["clf"].feature_importances_

feat_imp = sorted(
    zip(feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

print("\nTop 20 RF feature importances:")
for name, imp in feat_imp[:20]:
    print(f"{name}: {imp:.3f}")


RF Train accuracy: 1.0
RF Test accuracy : 1.0

Top 20 RF feature importances:
first_contact_after_experiencing_symptom_emergency_services: 0.191
first_contact_after_experiencing_symptom_doctor: 0.088
how_soon_treatment_should_be_taken_after_noticing_symptoms_immediately: 0.058
first_contact_after_experiencing_symptom_family/friend: 0.043
risk_factors_blood_pressure_or_heart_problem, diabetes_or_obesity, smoking_or_alcohol_or_drugs, age_or_family_history_of_stroke_gender, Unaware: 0.038
first_contact_after_experiencing_symptom_hospital: 0.034
where_to_go_after_experiencing_symptoms_of_brain_stroke_hospital_or_medical_centre: 0.034
risk_factors_Unaware: 0.029
if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?_Neurologist: 0.027
if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?_Physician: 0.022
stroke_symptoms_no_response: 0.022
stroke_symptoms_speech_or_confusion,paralysis_or_weakness,vision_problems,dizziness,headache: 0.019
w

In [12]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    n_jobs=-1
)

log_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", log_reg)
])

log_model.fit(X_train, y_train)

print("LogReg Train accuracy:", log_model.score(X_train, y_train))
print("LogReg Test accuracy :", log_model.score(X_test, y_test))


c:\Users\Harshali g\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg Train accuracy: 0.9957438184029185
LogReg Test accuracy : 0.9927066450567261


In [13]:
ohe = log_model.named_steps["preprocess"].named_transformers_["cat"]
feature_names = ohe.get_feature_names_out(categorical_features)

clf = log_model.named_steps["clf"]
classes = clf.classes_

coef = clf.coef_  # shape: (n_classes, n_features)

for class_idx, cls in enumerate(classes):
    print(f"\nTop features for class: {cls}")
    class_coefs = coef[class_idx]
    # sort by absolute value
    idx_sorted = class_coefs.argsort()[::-1][:15]
    for i in idx_sorted:
        print(f"{feature_names[i]}: {class_coefs[i]:.3f}")



Top features for class: High Awareness
first_contact_after_experiencing_symptom_emergency_services: 2.652
if_you_experience_symptoms_of_warning_signs,_which_specialist_would_you_consult?_Neurologist: 2.386
what_advice_would_you_give_for_someone_experiencing_stroke_symptoms_call_emergency_services: 2.108
where_to_go_after_experiencing_symptoms_of_brain_stroke_emergency_services: 1.840
stroke_symptoms_speech_or_confusion,paralysis_or_weakness,vision_problems,dizziness: 1.395
how_soon_treatment_should_be_taken_after_noticing_symptoms_immediately: 1.318
how_soon_would_you_consult_a_specialist_after_experiencing_the_first_symptom?_Immediately: 1.314
where_to_go_after_experiencing_symptoms_of_brain_stroke_hospital_or_medical_centre: 0.978
do_you_know_what_is_a_brain_stroke?_Yes: 0.802
risk_factors_blood_pressure_or_heart_problem, diabetes_or_obesity, smoking_or_alcohol_or_drugs: 0.737
what_advice_would_you_give_for_someone_experiencing_stroke_symptoms_other: 0.583
salary_2,00,001 - 4,00,000